# 🎬 Đếm trên VIDEO THẬT — GOOGLE COLAB & KAGGLE (chạy HẾT, tối ưu nhanh)

14 video (xe / người / dây chuyền), toạ độ vạch-vùng do **bạn tự vẽ**. Notebook chạy
**mỗi video 1 lần** (1 prompt) cho nhanh; xe/người dùng YOLO (nhanh), dây chuyền
open-vocab giới hạn ít frame.

> ▶️ Chạy **lần lượt từ trên xuống**. **Cell 1 tự nhận Colab/Kaggle.**
> ⚠️ **Bật GPU**: Colab → *Runtime → Change runtime type → T4 GPU*; Kaggle → *Settings → Accelerator → GPU*.


## 1) Tải code + cài thư viện (tự nhận diện Colab/Kaggle)

In [ ]:
import os
if os.path.isdir('/kaggle/working'):      WORK = '/kaggle/working'   # Kaggle
elif os.path.isdir('/content'):           WORK = '/content'          # Google Colab
else:                                      WORK = os.path.abspath('.')
os.makedirs(WORK, exist_ok=True); os.chdir(WORK)
print('📂 WORK =', WORK)

REPO   = 'https://github.com/nguyendinhhuyht20032004-ai/VisionOS.git'
BRANCH = 'claude/rebuild-visionos-codebase-tgg0mf'
if not os.path.isdir(f'{WORK}/VisionOS/.git'):
    os.system(f'git clone -q {REPO} {WORK}/VisionOS')
os.chdir(f'{WORK}/VisionOS')
os.system(f'git fetch -q origin {BRANCH} && git checkout -q {BRANCH} && git reset --hard -q origin/{BRANCH}')
os.chdir(f'{WORK}/VisionOS/VisionOS')
print('📁 cwd  =', os.getcwd())
os.system("pip install -q ultralytics 'supervision>=0.21' opencv-python-headless")

import torch
print('🖥️  GPU :', torch.cuda.get_device_name(0) if torch.cuda.is_available()
      else '❌ CHƯA BẬT GPU! Colab: Runtime → Change runtime type → T4 GPU · Kaggle: Settings → Accelerator → GPU')


## 2) ✅ Kiểm tra tải video (nhanh, không cần model)

In [ ]:
!python run_scenarios.py --download-only


## 3) ⭐ XEM TRƯỚC vạch/vùng mọi video (lưới % + vạch/vùng, không cần model)
Xem đặt đúng chưa; lệch thì vẽ lại ở cell dưới.

In [ ]:
import os, glob
from IPython.display import Image, display, Markdown
WORK = globals().get('WORK') or ('/kaggle/working' if os.path.isdir('/kaggle/working') else '/content')
!python run_scenarios.py --preview {WORK}/prev
allp = f'{WORK}/prev/_ALL.jpg'
if os.path.exists(allp):
    display(Markdown('### 🧩 TỔNG HỢP tất cả trường hợp')); display(Image(filename=allp, width=900))
for p in sorted(glob.glob(f'{WORK}/prev/*.jpg')):
    if os.path.basename(p) == '_ALL.jpg': continue
    display(Markdown(f'**{os.path.basename(p)}** — vàng=vạch, xanh=vùng, lưới=%')); display(Image(filename=p, width=760))


## 4) 🖊️ TỰ VẼ vạch/vùng bằng chuột (1 khung, nút ◀/▶ sang ảnh khác)
`draw_gallery("all")` = mọi video. VẠCH: 2 điểm → *Xong VẠCH*. VÙNG: ≥3 điểm → bấm
điểm đầu (đỏ) / bấm đúp / *Xong VÙNG*. Xong **📋 Copy tất cả** gửi Claude.

In [ ]:
import sys
for _m in [x for x in list(sys.modules) if x.startswith("recognition")]:
    del sys.modules[_m]
from recognition.draw_tool import draw_gallery
from IPython.display import HTML
HTML(draw_gallery("all"))


## 5) ⏳ Tải model 3B TRƯỚC (cho dây chuyền open-vocab, chạy 1 lần)
Dây chuyền dùng LocateAnything-3B (~6GB). Chạy cell này 1 lần, **ĐỪNG bấm Stop**
(3–8 phút). `KeyboardInterrupt` = bị ngắt, không phải lỗi.

In [ ]:
from huggingface_hub import snapshot_download
print("✅ Model ở cache:", snapshot_download("nvidia/LocateAnything-3B"))


## 6) ▶️ CHẠY ĐẾM HẾT — mỗi video 1 lần (tối ưu nhanh)
Đã dùng **YOLOv8m** (mạnh hơn nano nhiều — bắt người ở xa/tối + xe nhỏ tốt hơn),
`conf 0.25`, `imgsz 960`. Xe top-down thêm `--imgsz 1280` (xe nhỏ). Vật vẫn bị SÓT?
Tăng model: thêm `--yolo-weights yolov8l.pt` (hoặc `yolov8x.pt`, chậm hơn) / hạ `--confidence 0.2`.

> 🧮 Đếm bằng **supervision (ByteTrack + LineZone/PolygonZone)** — tracker mạnh, đếm cắt vạch/vùng chuẩn hơn (tự bật vì đã cài supervision). Ép bộ tự viết: thêm `--engine builtin`.


In [ ]:
WORK = globals().get('WORK') or '/content'
# 🚗 XE (4 video: 3 vạch + veh_px2 đếm 5 vùng) — YOLOv8m, imgsz 1280 (xe nhỏ top-down)
!python run_scenarios.py --task vehicles --max-frames 200 --imgsz 1280 --save-dir {WORK}/scen_out


In [ ]:
WORK = globals().get('WORK') or '/content'
# 🚁 XE TOP-DOWN — yolov8x + --tile + imgsz 1280 (xe nhìn từ trên rất nhỏ). Chậm nhưng bắt được.
!python run_scenarios.py --task vehicles --only 'giao thông' --tile --yolo-weights yolov8x.pt --imgsz 1280 --confidence 0.2 --save-dir {WORK}/scen_out


In [ ]:
WORK = globals().get('WORK') or '/content'
# 🚶 NGƯỜI — bản MẠNH: yolov8x + --tile (toàn ảnh + cắt ô) + conf 0.2 → bắt đám đông,
# người ở XA / bị che tốt hơn HẲN yolov8m. Chậm hơn; giảm --max-frames nếu cần.
!python run_scenarios.py --task people --tile --yolo-weights yolov8x.pt --confidence 0.2 --max-frames 150 --save-dir {WORK}/scen_out
# Nhanh hơn (kém chính xác hơn): bỏ --tile, hoặc --yolo-weights yolov8m.pt


In [ ]:
# 📦 DÂY CHUYỀN (milk YOLO + 4 video open-vocab) — ít frame cho nhanh
!python run_scenarios.py --task conveyor --max-frames 60 --save-dir {WORK}/scen_out


## 7) (tuỳ chọn) Query suite LITE — test nhiều mô tả trên 1 video (chậm hơn)
Bỏ `#` để chạy. `--suite` = LITE (1–2 query/nhóm). Nên chạy 1 video.

In [ ]:
WORK = globals().get('WORK') or '/content'
# !python run_scenarios.py --task conveyor --only pkg --suite --save-dir {WORK}/scen_out
# !python run_scenarios.py --task people --only walk --suite --save-dir {WORK}/scen_out


## 8) 🎥 Xem / tải video output

In [ ]:
import glob, os
from IPython.display import Video, display
WORK = globals().get('WORK') or ('/kaggle/working' if os.path.isdir('/kaggle/working') else '/content')
vids = sorted(glob.glob(f'{WORK}/scen_out/**/*.mp4', recursive=True))
print(f'{len(vids)} video output:')
for p in vids: print('  ', p)
if vids:
    src = vids[0]; dst = f'{WORK}/preview_h264.mp4'
    os.system(f'ffmpeg -y -loglevel error -i "{src}" -vcodec libx264 -pix_fmt yuv420p "{dst}"')
    print('Xem:', src); display(Video(dst, embed=True, width=700))


---
### Ghi chú
- Chạy **mỗi video 1 prompt** cho nhanh (không phải bộ query đầy đủ). Muốn test sâu → mục 7.
- `veh_px2` đếm **5 vùng gộp 1 số**; siêu thị đếm **3 vùng gộp**. Video output: vàng=vạch, xanh=vùng.
- Đặt lại vạch/vùng: vẽ ở mục 4 rồi gửi toạ độ cho Claude (đưa vào `recognition/video_catalog.py`).